In [1]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tabicl import TabICLClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import precision_recall_curve
import itertools
import sys
# Get the current working directory of the notebook
current_dir = os.getcwd()
# Get the parent directory
parent_dir = os.path.abspath(os.path.join(current_dir, os.pardir))
# Add the parent directory to sys.path
sys.path.insert(0, parent_dir)
from imputation import * 
from internal_test import *
from summary_table import *
from feature_importance import *
# Show all rows
pd.set_option('display.max_rows', None)
# Show all columns
pd.set_option('display.max_columns', None)
# Do not truncate column values
pd.set_option('display.max_colwidth', None)

In [2]:
year = '2000-2006'

In [3]:
# Define input columns and target
X_columns = [
    #'hhidpn', 'NIWWAVE','demcls',
     'child', 'lbrf', 'shlt', 'ageym', 'height', 'weight', 'smokev', 'proxy', 'effort', 'hibpe', 'diabe', 'vgactx', 'slfmem', 'livpar', 'momage', 'dadage', 'livsib', 'hlthlm', 'hosp', 'nrshom', 'nrstim', 'nrsnit', 'doctim', 'depres', 'sleepr', 'whappy', 'flone', 'fsad', 'going', 'enlife', 'drink', 'smoken', 'cancre', 'lunge', 'hearte', 'stroke', 'arthre', 'toilta', 'adl5a', 'mapa', 'walksa', 'walk1a', 'sita', 'chaira', 'climsa', 'clim1a', 'stoopa', 'lifta', 'dimea', 'armsa', 'pusha', 'mobila', 'lgmusa', 'grossa', 'finea', 'gender', 'edstg2 8to11', 'edstg3 12', 'edstg4 13above', 'cendiv2 mid atlantic', 'cendiv3 en central', 'cendiv4 wn central', 'cendiv5 s atlantic', 'cendiv6 es central', 'cendiv7 ws central', 'cendiv8 mountain', 'cendiv9 pacific', 'cendiv11 not us or inc us terr', 'mstat2 married spouse absent', 'mstat3 partnered', 'mstat4 separated', 'mstat5 divorced', 'mstat7 widowed', 'mstat8 never married', 'raceeth1 ', 'raceeth2 ', 'raceeth3 '
]

y_column = 'y'

# Directories and file paths
grid_search_dir = f'../data/{year}'

In [4]:
# Define grid search parameter grid
param_grid = {
    'n_estimators': [16, 32],
    'batch_size': [8, 16]
}

In [52]:
imputation_methods = ['none','mean','mice']  

In [53]:
final_rows, X_test, y_test, final_model = internal_test(year,TabICLClassifier,param_grid,imputation_methods,grid_search_dir,X_columns, y_column)


Internal Test (2000-2006)
Model: <class 'tabicl.sklearn.classifier.TabICLClassifier'>
Protocol: 10-Fold Cross-Validation with 8:1:1 Train : Validation : Test splits
Metrics are reported as percentages with two decimal places.


Processing imputation method: none
For imputation method 'none', best candidate: (16, 8) with average validation AUPR = 23.09%
Final test metrics for imputation method 'none': AUC = 83.47%, ACC = 96.49%, AUPR = 21.44%
Standard deviations across folds (for error bars): AUC_std = 2.53%, ACC_std = 0.30%, AUPR_std = 2.49%

Processing imputation method: mean
For imputation method 'mean', best candidate: (16, 8) with average validation AUPR = 23.95%
Final test metrics for imputation method 'mean': AUC = 83.93%, ACC = 96.49%, AUPR = 21.99%
Standard deviations across folds (for error bars): AUC_std = 2.45%, ACC_std = 0.30%, AUPR_std = 3.42%

Processing imputation method: mice
For imputation method 'mice', best candidate: (16, 8) with average validation AUPR = 24.81%
Fi

#### Internal Test with stratified data splitting via miceforest imputation
##### Train:Val:Test = 6:2:2

In [5]:
df_full = pd.read_csv('../data/preprocessed data/2000(00-06y)encode.csv')
X_full, y_full = impute_data_and_y(df_full, 'mice', X_columns, y_column)

In [6]:
train_sets = []
train_val_sets = []
val_sets = []
test_sets = []
for i in range(10):
    # make train+val and test data split 8:2
    X_train_val, X_test, y_train_val, y_test = train_test_split(X_full, y_full, test_size=0.2, stratify=y_full, random_state=i)
    train_val_sets.append((X_train_val, y_train_val))
    test_sets.append((X_test, y_test))
    # make train and val data split 6:2. finally train:val:test = 6:2:2
    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=i)
    train_sets.append((X_train, y_train))
    val_sets.append((X_val, y_val))

In [7]:
#grid search
candidate_results = {}
for train, val in zip(train_sets,val_sets ):
    X_train, y_train = train
    X_val, y_val = val
    combinations = itertools.product(*param_grid.values())
    for parameter in combinations:
        candidate = list(param_grid.keys())
        try:
            m = TabICLClassifier(**dict(zip(candidate, parameter)),use_amp=True, checkpoint_version='tabicl-classifier-v1.1-0506.ckpt')
            m.fit(X_train, y_train)
            y_val_prob = m.predict_proba(X_val)[:, 1]
            aupr = average_precision_score(y_val, y_val_prob)
            candidate_results.setdefault(parameter, []).append(aupr)
        except Exception:
            continue
avg_val_aupr = {cand: np.mean(scores) for cand, scores in candidate_results.items()}
best_candidate = max(avg_val_aupr, key=avg_val_aupr.get)
best_val_aupr = avg_val_aupr[best_candidate]
print(
    f"For imputation method 'mice', best candidate: {best_candidate} "
    f"with average validation AUPR = {pct(best_val_aupr)}"
        )

For imputation method 'mice', best candidate: (16, 8) with average validation AUPR = 31.12%


In [8]:
per_fold = []
optimal_thresholds_list = []

# Step 1: Find optimal threshold for each fold
for train_val, test in zip(train_val_sets, test_sets):
    X_train_val, y_train_val = train_val
    X_test, y_test = test
    final_model = TabICLClassifier(**dict(zip(candidate, best_candidate)))
    final_model.fit(X_train_val, y_train_val)
    y_prob = final_model.predict_proba(X_test)[:, 1]
    
    # Find optimal threshold based on F1 score
    precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5
    
    optimal_thresholds_list.append(optimal_threshold)

# Step 2: Use mean optimal threshold for all folds
mean_optimal_threshold = np.mean(optimal_thresholds_list)
print(f"Mean Optimal Threshold: {mean_optimal_threshold:.4f}")
print(f"Optimal thresholds per fold: {[f'{t:.4f}' for t in optimal_thresholds_list]}")

# Step 3: Calculate metrics using optimal threshold
per_fold = []
for train_val, test in zip(train_val_sets, test_sets):
    X_train_val, y_train_val = train_val
    X_test, y_test = test
    final_model = TabICLClassifier(**dict(zip(candidate, best_candidate)))
    final_model.fit(X_train_val, y_train_val)
    y_prob = final_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= mean_optimal_threshold).astype(int)
    
    per_fold.append({
        "AUC": roc_auc_score(y_test, y_prob),
        "ACC": accuracy_score(y_test, y_pred),
        "AUPR": average_precision_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
    })

# Step 4: Calculate means and stds across folds
aucs      = np.array([m["AUC"]       for m in per_fold])
accs      = np.array([m["ACC"]       for m in per_fold])
auprs     = np.array([m["AUPR"]      for m in per_fold])
precisions = np.array([m["Precision"] for m in per_fold])
recalls   = np.array([m["Recall"]    for m in per_fold])
f1s       = np.array([m["F1"]        for m in per_fold])

mean_auc,  std_auc  = aucs.mean(),  aucs.std()
mean_acc,  std_acc  = accs.mean(),  accs.std()
mean_aupr, std_aupr = auprs.mean(), auprs.std()
mean_precision, std_precision = precisions.mean(), precisions.std()
mean_recall,    std_recall    = recalls.mean(),    recalls.std()
mean_f1,        std_f1        = f1s.mean(),        f1s.std()

print("\nTabICLClassifier with MICE Imputation — Internal Test")
print(f"Optimal Threshold Used: {mean_optimal_threshold:.4f}")
print(
    f"Final test metrics: "
    f"AUC = {pct(mean_auc)}, ACC = {pct(mean_acc)}, AUPR = {pct(mean_aupr)}, "
    f"Precision = {pct(mean_precision)}, Recall = {pct(mean_recall)}, F1 = {pct(mean_f1)}"
)
print(
    f"Standard deviations across folds: "
    f"AUC_std = {pct(std_auc)}, ACC_std = {pct(std_acc)}, AUPR_std = {pct(std_aupr)}, "
    f"Precision_std = {pct(std_precision)}, Recall_std = {pct(std_recall)}, F1_std = {pct(std_f1)}"
)

Mean Optimal Threshold: 0.1785
Optimal thresholds per fold: ['0.1458', '0.1469', '0.2014', '0.1582', '0.1750', '0.1631', '0.1930', '0.1723', '0.1986', '0.2309']

TabICLClassifier with MICE Imputation — Internal Test
Optimal Threshold Used: 0.1785
Final test metrics: AUC = 82.38%, ACC = 85.68%, AUPR = 32.04%, Precision = 32.07%, Recall = 49.56%, F1 = 38.92%
Standard deviations across folds: AUC_std = 0.83%, ACC_std = 0.60%, AUPR_std = 1.21%, Precision_std = 1.21%, Recall_std = 1.33%, F1_std = 0.93%


In [ ]:
def permutation_importance_with_progress(model, X, y, n_repeats=10, scoring=accuracy_score):
    baseline_score = scoring(y, model.predict(X))
    importances = []
    
    # Convert to numpy if it's a DataFrame
    is_dataframe = isinstance(X, pd.DataFrame)
    feature_names = X.columns.tolist() if is_dataframe else [f"feature_{i}" for i in range(X.shape[1])]
    X_array = X.values if is_dataframe else X
    
    # Iterate through features with progress bar
    for col_idx in tqdm(range(X_array.shape[1]), desc="Computing feature importance"):
        feature_importances = []
        
        for _ in range(n_repeats):
            X_permuted = X_array.copy()
            # Permute the feature
            X_permuted[:, col_idx] = np.random.permutation(X_permuted[:, col_idx])
            
            # Convert back to DataFrame if original was DataFrame
            if is_dataframe:
                X_permuted_df = pd.DataFrame(X_permuted, columns=feature_names, index=X.index)
                permuted_score = scoring(y, model.predict(X_permuted_df))
            else:
                permuted_score = scoring(y, model.predict(X_permuted))
            
            # Importance is the decrease in score
            feature_importances.append(baseline_score - permuted_score)
        
        importances.append({
            'feature': feature_names[col_idx],
            'mean': np.mean(feature_importances),
            'std': np.std(feature_importances)
        })
    
    return importances
# Usage
# importances = permutation_importance_with_progress(final_model, X_test, y_test,scoring=average_precision_score)

In [ ]:
X_test.reset_index(drop=True, inplace=True)
y_test.reset_index(drop=True, inplace=True)

In [ ]:
importances_df = pd.DataFrame(importances)
importances_df = importances_df.sort_values('mean', ascending=False)
print("Top-5 Grouped Permutation-Based Feature Importances (ΔAUPR)\nTabICLClassifier with MICE Imputation — Internal Test (2000–2006)")
print(importances_df.head(5).round(3))

Top-5 Grouped Permutation-Based Feature Importances (ΔAUPR)
TabICLClassifier with MICE Imputation — Internal Test (2000–2006)
               feature   mean    std
3                ageym  0.004  0.002
58      edstg4 13above  0.004  0.002
12              slfmem  0.003  0.002
74           raceeth1   0.003  0.002
63  cendiv6 es central  0.002  0.000
